# Lab | Tools prompting

**Replace the existing two tools decorators, by creating 3 new ones and adjust the prompts accordingly**

### How to add ad-hoc tool calling capability to LLMs and Chat Models

:::{.callout-caution}

Some models have been fine-tuned for tool calling and provide a dedicated API for tool calling. Generally, such models are better at tool calling than non-fine-tuned models, and are recommended for use cases that require tool calling. Please see the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide for more information.

In this guide, we'll see how to add **ad-hoc** tool calling support to a chat model. This is an alternative method to invoke tools if you're using a model that does not natively support tool calling.

We'll do this by simply writing a prompt that will get the model to invoke the appropriate tools. Here's a diagram of the logic:

<br>

![chain](https://education-team-2020.s3.eu-west-1.amazonaws.com/ai-eng/tool_chain.svg)

## Setup

We'll need to install the following packages:

In [1]:
import sys
!{sys.executable} -m
%pip install --upgrade --quiet langchain langchain-community langchain_openai

Argument expected for the -m option
usage: /Library/Frameworks/Python.framework/Versions/3.11/Resources/Python.app/Contents/MacOS/Python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Try `python -h' for more information.

[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


If you'd like to use LangSmith, uncomment the below:

In [2]:
import getpass
import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = getpass.getpass()

You can select any of the given models for this how-to guide. Keep in mind that most of these models already [support native tool calling](https://python.langchain.com/docs/integrations/chat), so using the prompting strategy shown here doesn't make sense for these models, and instead you should follow the [how to use a chat model to call tools](https://python.langchain.com/docs/how_to/tool_calling/) guide.

```{=mdx}
import ChatModelTabs from "@theme/ChatModelTabs";

<ChatModelTabs openaiParams={`model="gpt-4"`} />
```

To illustrate the idea, we'll use `phi3` via Ollama, which does **NOT** have native support for tool calling. If you'd like to use `Ollama` as well follow [these instructions](https://python.langchain.com/docs/integrations/chat/ollama).

In [3]:
from langchain_community.llms import Ollama

model = Ollama(model="phi3")

/var/folders/rn/hjmgpq6j4bl5hk8kkpw6k1f40000gn/T/ipykernel_18156/1427064109.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  model = Ollama(model="phi3")



#  How to Install and Run Ollama with the Phi-3 Model

This guide walks you through installing **Ollama** and running the **Phi-3** model on Windows, macOS, and Linux.

---

## Windows

1. **Download Ollama for Windows**  
   Go to: [https://ollama.com/download](https://ollama.com/download)  
   Download and run the installer.

2. **Verify Installation**  
   Open **Command Prompt** and type:
   ```bash
   ollama --version
   ```

3. **Run the Phi-3 Model**  
   In the same terminal:
   ```bash
   ollama run phi3
   ```

4. **If you get a CUDA error (GPU memory issue)**  
   Run Ollama in **CPU mode**:
   ```bash
   set OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  macOS

1. **Install via Homebrew**  
   Open the Terminal and run:
   ```bash
   brew install ollama
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Linux

1. **Install Ollama**  
   Open a terminal and run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. **Run the Phi-3 Model**
   ```bash
   ollama run phi3
   ```

3. **To force CPU mode (no GPU)**
   ```bash
   export OLLAMA_NO_CUDA=1
   ollama run phi3
   ```

---

##  Notes

- The first time you run `ollama run phi3`, it will **download the model**, so make sure you’re connected to the internet.
- Once downloaded, it works **offline**.
- Keep the terminal open and running in the background while using Ollama from your code or notebook.


## Create a tool

First, let's create an `add` and `multiply` tools. For more information on creating custom tools, please see [this guide](https://python.langchain.com/docs/how_to/custom_tools/).

In [4]:
from langchain_core.tools import tool
import math


@tool
def power(base: float, exponent: float) -> float:
    """Raise `base` to the power of `exponent` and return the result."""
    return base ** exponent


@tool
def count_vowels(text: str) -> int:
    """Count the number of vowels (a, e, i, o, u) in the given text."""
    return sum(1 for ch in text.lower() if ch in "aeiou")


@tool
def convert_celsius_to_fahrenheit(celsius: float) -> float:
    """Convert a temperature in Celsius to Fahrenheit."""
    return celsius * 9 / 5 + 32


tools = [power, count_vowels, convert_celsius_to_fahrenheit]

# Let's inspect the tools
for t in tools:
    print("--")
    print(t.name)
    print(t.description)
    print(t.args)


--
power
Raise `base` to the power of `exponent` and return the result.
{'base': {'title': 'Base', 'type': 'number'}, 'exponent': {'title': 'Exponent', 'type': 'number'}}
--
count_vowels
Count the number of vowels (a, e, i, o, u) in the given text.
{'text': {'title': 'Text', 'type': 'string'}}
--
convert_celsius_to_fahrenheit
Convert a temperature in Celsius to Fahrenheit.
{'celsius': {'title': 'Celsius', 'type': 'number'}}


In [5]:
power.invoke({"base": 2, "exponent": 10})


1024.0

## Creating our prompt

We'll want to write a prompt that specifies the tools the model has access to, the arguments to those tools, and the desired output format of the model. In this case we'll instruct it to output a JSON blob of the form `{"name": "...", "arguments": {...}}`.

In [6]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import render_text_description

rendered_tools = render_text_description(tools)
print(rendered_tools)


power(base: float, exponent: float) -> float - Raise `base` to the power of `exponent` and return the result.
count_vowels(text: str) -> int - Count the number of vowels (a, e, i, o, u) in the given text.
convert_celsius_to_fahrenheit(celsius: float) -> float - Convert a temperature in Celsius to Fahrenheit.


In [7]:
system_prompt = f"""\
You are an assistant with access to the following tools. \
Use them to answer the user. Here are the tools and their descriptions:

{rendered_tools}

Given the user input, choose exactly ONE tool and return its name and arguments
as a single JSON blob with the keys 'name' and 'arguments'.

`arguments` must be a dictionary mapping each argument name (as declared by the
tool) to the value to pass. Numeric arguments must be JSON numbers, text
arguments must be JSON strings. Do not add extra keys, comments, or prose
around the JSON.
"""

prompt = ChatPromptTemplate.from_messages(
    [("system", system_prompt), ("user", "{input}")]
)


In [8]:
chain = prompt | model
message = chain.invoke({"input": "What is 2 to the power of 10?"})

# Let's take a look at the output from the model
# if the model is an LLM (not a chat model), the output will be a string.
if isinstance(message, str):
    print(message)
else:  # Otherwise it's a chat model
    print(message.content)


```json
{
  "name": "power",
  "arguments": {
    "base": 2.0,
    "exponent": 10.0
  }
}
```
Asigne this task to the tool `power(base: float, exponent: float)` with arguments base set as `2.0` and exponent set as `10.0`. The result of using this command is \(2^{10}\), which equals 1024.


## Adding an output parser

We'll use the `JsonOutputParser` for parsing our models output to JSON.

In [9]:
from langchain_core.output_parsers import JsonOutputParser

chain = prompt | model | JsonOutputParser()
chain.invoke({"input": "How many vowels are in the sentence 'Hello beautiful world'?"})


{'name': 'count_vowels', 'arguments': {'text': 'Hello beautiful world'}}

:::{.callout-important}

🎉 Amazing! 🎉 We now instructed our model on how to **request** that a tool be invoked.

Now, let's create some logic to actually run the tool!
:::

## Invoking the tool 🏃

Now that the model can request that a tool be invoked, we need to write a function that can actually invoke 
the tool.

The function will select the appropriate tool by name, and pass to it the arguments chosen by the model.

In [10]:
from typing import Any, Dict, Optional, TypedDict

from langchain_core.runnables import RunnableConfig


class ToolCallRequest(TypedDict):
    """A typed dict that shows the inputs into the invoke_tool function."""

    name: str
    arguments: Dict[str, Any]


def invoke_tool(
    tool_call_request: ToolCallRequest, config: Optional[RunnableConfig] = None
):
    """A function that we can use the perform a tool invocation.

    Args:
        tool_call_request: a dict that contains the keys name and arguments.
            The name must match the name of a tool that exists.
            The arguments are the arguments to that tool.
        config: This is configuration information that LangChain uses that contains
            things like callbacks, metadata, etc.See LCEL documentation about RunnableConfig.

    Returns:
        output from the requested tool
    """
    tool_name_to_tool = {tool.name: tool for tool in tools}
    name = tool_call_request["name"]
    requested_tool = tool_name_to_tool[name]
    return requested_tool.invoke(tool_call_request["arguments"], config=config)

Let's test this out 🧪!

In [11]:
invoke_tool({"name": "convert_celsius_to_fahrenheit", "arguments": {"celsius": 25}})


77.0

## Let's put it together

Let's put it together into a chain that creates a calculator with add and multiplication capabilities.

In [12]:
chain = prompt | model | JsonOutputParser() | invoke_tool
chain.invoke({"input": "Convert 100 degrees Celsius to Fahrenheit"})


212.0

## Returning tool inputs

It can be helpful to return not only tool outputs but also tool inputs. We can easily do this with LCEL by `RunnablePassthrough.assign`-ing the tool output. This will take whatever the input is to the RunnablePassrthrough components (assumed to be a dictionary) and add a key to it while still passing through everything that's currently in the input:

In [13]:
from langchain_core.runnables import RunnablePassthrough

chain = (
    prompt | model | JsonOutputParser() | RunnablePassthrough.assign(output=invoke_tool)
)
chain.invoke({"input": "What is 7 raised to the power of 3?"})


{'name': 'power', 'arguments': {'base': 7.0, 'exponent': 3.0}, 'output': 343.0}

## What's next?

This how-to guide shows the "happy path" when the model correctly outputs all the required tool information.

In reality, if you're using more complex tools, you will start encountering errors from the model, especially for models that have not been fine tuned for tool calling and for less capable models.

You will need to be prepared to add strategies to improve the output from the model; e.g.,

1. Provide few shot examples.
2. Add error handling (e.g., catch the exception and feed it back to the LLM to ask it to correct its previous output).

In [21]:
SYSTEM_TEMPLATE = """You are an assistant with access to the following tools. \
Use them to answer the user. Here are the tools and their descriptions:

""" + rendered_tools + """

Given the user input, choose exactly ONE tool and return its name and arguments
as a single JSON blob with the keys 'name' and 'arguments'. Numeric arguments
must be JSON numbers, text arguments must be JSON strings. Do not add prose
around the JSON.

Examples:

User: What is 5 to the power of 3?
Output: {{"name": "power", "arguments": {{"base": 5, "exponent": 3}}}}

User: How many vowels are in "open ai"?
Output: {{"name": "count_vowels", "arguments": {{"text": "open ai"}}}}

User: Convert 0 degrees Celsius to Fahrenheit.
Output: {{"name": "convert_celsius_to_fahrenheit", "arguments": {{"celsius": 0}}}}
"""

few_shot_prompt = ChatPromptTemplate.from_messages(
    [("system", SYSTEM_TEMPLATE), ("user", "{input}")]
)

few_shot_chain = few_shot_prompt | model | JsonOutputParser() | invoke_tool
few_shot_chain.invoke({"input": "What is 2 to the power of 8?"})

256.0

In [22]:
from langchain_core.exceptions import OutputParserException


def robust_invoke(user_input: str, max_retries: int = 2):
    """Run the tool-calling chain with automatic self-correction on failure."""
    last_error = None
    last_raw   = None

    for attempt in range(max_retries + 1):
        try:
            if attempt == 0:
                return few_shot_chain.invoke({"input": user_input})
            else:
                correction = (
                    f"Your previous answer to the question "
                    f"\"{user_input}\" was:\n{last_raw}\n\n"
                    f"It failed with this error: {last_error}\n"
                    f"Please return ONLY a valid JSON blob with keys "
                    f"'name' and 'arguments' that matches one of the tools."
                )
                fixed_chain = few_shot_prompt | model | JsonOutputParser() | invoke_tool
                return fixed_chain.invoke({"input": correction})

        except (OutputParserException, KeyError, ValueError, TypeError) as e:
            last_error = str(e)
            try:
                last_raw = (few_shot_prompt | model).invoke({"input": user_input})
                last_raw = last_raw.content if hasattr(last_raw, "content") else last_raw
            except Exception:
                last_raw = "<no output>"
            print(f"[attempt {attempt + 1}] failed: {e}")

    raise RuntimeError(
        f"Tool call failed after {max_retries + 1} attempts. "
        f"Last error: {last_error}"
    )


print("--- Easy question ---")
print(robust_invoke("What is 4 to the power of 5?"))

print("\n--- Tricky question ---")
print(robust_invoke("How hot is 37 degrees Celsius in Fahrenheit?"))

--- Easy question ---
1024.0

--- Tricky question ---
98.6
